# Earthquake Directionality — Google Colab Runner

# Mount Drive, install dependencies, and run the modular project. 

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

# Project paths

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Earthquake_ML_MiniProject")
PROJECT = ROOT / "earthquake-ground-motion-directionality"

AT2_DIR = ROOT / "UnscaledAccRecord8878"
FLATFILE = ROOT / "Updated_NGA_West2_flatfiles_part1" / (
    "Updated_NGA_West2_Flatfile_RotD50_d005_public_version.xlsx"
)

OUT = PROJECT / "outputs"

assert PROJECT.exists(), f"Project folder not found: {PROJECT}"
assert AT2_DIR.exists(), f"AT2 folder not found: {AT2_DIR}"
assert FLATFILE.exists(), f"Flatfile not found: {FLATFILE}"

print("Project:", PROJECT)
print("AT2 directory:", AT2_DIR)
print("Flatfile:", FLATFILE)
print("Output directory:", OUT)

# Pipeline execution cell

In [ ]:
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        str(PROJECT / "run.py"),
        "--at2", str(AT2_DIR),
        "--flatfile", str(FLATFILE),
        "--out", str(OUT),
        "--max-km", "100",
    ],
    check=True,
)

# Paper-informed experiment

In [ ]:
experiment_script = PROJECT / "paper_aligned_experiment.py"

if experiment_script.exists():
    subprocess.run(
        [
            sys.executable,
            str(experiment_script),
            "--out", str(OUT),
        ],
        check=True,
    )
else:
    print("paper_aligned_experiment.py is not present in the project root.")

# Output inspection cell

In [ ]:
import json
import pandas as pd

summary_file = OUT / "summary.json"

if summary_file.exists():
    print(json.dumps(json.loads(summary_file.read_text()), indent=2))
else:
    print("summary.json not found. Check whether the pipeline completed.")

for filename in [
    "orientations.csv",
    "records_with_metadata.csv",
    "station_pairs.csv",
    "same_event_pairs.csv",
]:
    filepath = OUT / filename
    if filepath.exists():
        print(f"\n{filename}")
        display(pd.read_csv(filepath).head())